# 中国货币政策文本课设最终复现 Notebook

本 Notebook 展示固定路线下的数据处理、文本特征工程、分组交叉验证、金融事件研究、稳健性检验和提交包审计。所有数字均由代码读取正式结果文件。

In [1]:
from pathlib import Path
import json
import pandas as pd
ROOT = Path.cwd()
while not (ROOT / 'configs/project.yml').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
import sys
sys.path.insert(0, str(ROOT))
print(ROOT)

D:\PyCharm\Quant\monetary_policy_project


## 1. 锁定分析计划与正式样本

In [2]:
from src.monetary_policy.sample import verify_final_analysis_plan, sample_bounds, is_in_formal_sample
verify_final_analysis_plan()
features = pd.read_csv(ROOT / 'data/processed/refactor_text_features.csv')
print('sample_bounds =', sample_bounds())
print(features[['report_id','report_period','in_formal_sample']].tail())
print('formal_n =', int(features['in_formal_sample'].sum()))

sample_bounds = ('2006Q1', '2025Q4')
   report_id report_period  in_formal_sample
76    2025Q1        2025Q1              True
77    2025Q2        2025Q2              True
78    2025Q3        2025Q3              True
79    2025Q4        2025Q4              True
80    2026Q1        2026Q1             False
formal_n = 80


## 2. 数据来源、章节修复与可追溯性

In [3]:
registry = pd.read_csv(ROOT / 'data/source_registry.csv')
display(registry[['dataset_name','source_organization','coverage_start','coverage_end','license_or_terms']])
repair_path = ROOT / 'output/diagnostics/section_repair_report.xlsx'
if repair_path.exists():
    display(pd.read_excel(repair_path))
else:
    print('section repair diagnostics not packaged; run_all.py --offline regenerates them')

,dataset_name,source_organization,coverage_start,coverage_end,license_or_terms
0,pbc_report_2006Q1,中国人民银行,2006Q1,2006Q1,PBC public website; redistribution terms not s...
1,pbc_report_2006Q2,中国人民银行,2006Q2,2006Q2,PBC public website; redistribution terms not s...
2,pbc_report_2006Q3,中国人民银行,2006Q3,2006Q3,PBC public website; redistribution terms not s...
3,pbc_report_2006Q4,中国人民银行,2006Q4,2006Q4,PBC public website; redistribution terms not s...
4,pbc_report_2007Q1,中国人民银行,2007Q1,2007Q1,PBC public website; redistribution terms not s...
...,...,...,...,...,...
79,pbc_report_2025Q4,中国人民银行,2025Q4,2025Q4,PBC public website; redistribution terms not s...
80,pbc_report_2026Q1,中国人民银行,2026Q1,2026Q1,PBC public website; redistribution terms not s...
81,chinabond_government_yield_full,中央国债登记结算有限责任公司/中债估值中心,2006-03-01,2026-06-18,Public query/interface; redistribution not ass...
82,csi300_daily_full,第三方财经接口，经 AkShare 获取,2002-01-04,2026-06-18,Public query/interface; redistribution not ass...


,report_id,old_found,old_char_count,new_found,new_char_count,extraction_rule,manual_override,review_note
0,2006Q1,False,0,True,1914,manual_title_alias_last_match,True,正文第五部分第二节标题，目录中同名标题不使用；取最后一次匹配。
1,2006Q4,False,0,True,1701,manual_title_alias_last_match,True,正文第五部分第二节标题，取最后一次匹配。
2,2007Q2,False,0,True,1072,manual_title_alias_last_match,True,正文第五部分第二节标题，取最后一次匹配。
3,2007Q4,False,0,True,2252,manual_title_alias_last_match,True,正文第五部分第三节标题，取最后一次匹配。


## 3. 当前词典实时重打分与人工标注验证

In [4]:
metrics_path = ROOT / 'output/diagnostics/text_validation_metrics.xlsx'
if metrics_path.exists():
    display(pd.read_excel(metrics_path, sheet_name='summary').T)
else:
    text_model = json.loads((ROOT / 'output/results/text_model_evaluation.json').read_text(encoding='utf-8'))
    display(pd.DataFrame([{k: v for k, v in obj.items() if k in ['n','n_groups','accuracy','macro_f1']} for obj in text_model.values() if isinstance(obj, dict) and 'accuracy' in obj]))

,0
total_sentences,240
annotation_path,D:\PyCharm\Quant\monetary_policy_project\data\...
annotation_sha256,11c3ecee9ac91720b5a7fba3d5986f0ea2245b36c3e9b9...
lexicon_path,D:\PyCharm\Quant\monetary_policy_project\data\...
lexicon_sha256,bc6342c17e37ab0695a1f9113f8ead2dca2f17350865ac...
completeness,"{'manual_sentiment_label': {'null': 0, 'empty_..."
illegal_labels,"{'sentiment': [], 'stance': [], 'topic': []}"
sentiment_accuracy,0.295833
sentiment_macro_f1,0.327196
sentiment_weighted_f1,0.23448


## 4. 字符 TF-IDF + LinearSVC 分组交叉验证

In [5]:
text_model = json.loads((ROOT / 'output/results/text_model_evaluation.json').read_text(encoding='utf-8'))
rows = []
for name, obj in text_model.items():
    if isinstance(obj, dict) and 'accuracy' in obj:
        rows.append({'task': name, 'n': obj.get('n'), 'groups': obj.get('n_groups'), 'accuracy': obj.get('accuracy'), 'macro_f1': obj.get('macro_f1')})
display(pd.DataFrame(rows))

,task,n,groups,accuracy,macro_f1
0,sentiment_cv,240,58,0.895833,0.799286
1,policy_stance_cv,240,58,0.858333,0.680429
2,policy_direction_cv,66,55,0.787879,0.730089
3,topic_cv,240,58,0.720833,0.332910


## 5. 学习曲线

In [6]:
lc = pd.read_excel(ROOT / 'output/tables/table_learning_curve_summary.xlsx')
display(lc)
display(lc.sort_values(['task','train_ratio']).groupby('task').tail(1))

,task,train_ratio,accuracy,macro_f1,n
0,sentiment,0.25,0.850000,0.647559,240
1,sentiment,0.40,0.879167,0.735517,240
2,sentiment,0.60,0.866667,0.732171,240
3,sentiment,0.80,0.891667,0.788745,240
4,sentiment,1.00,0.895833,0.799286,240
5,policy_stance,0.25,0.770833,0.431243,240
6,policy_stance,0.40,0.783333,0.489964,240
7,policy_stance,0.60,0.845833,0.662898,240
8,policy_stance,0.80,0.866667,0.696334,240
9,policy_stance,1.00,0.858333,0.680429,240


,task,train_ratio,accuracy,macro_f1,n
9,policy_stance,1.0,0.858333,0.680429,240
4,sentiment,1.0,0.895833,0.799286,240
14,topic,1.0,0.720833,0.332910,240


## 6. 扩展窗口 TF-IDF 创新度与连续主题关注度

In [7]:
topic = pd.read_csv(ROOT / 'data/processed/continuous_topic_attention.csv')
display(features[['report_id','guidance_similarity_expanding_tfidf','guidance_novelty','guidance_novelty_expanding_tfidf','fulltext_novelty_expanding_tfidf']].tail())
display(topic.loc[topic['in_formal_sample'], [c for c in topic.columns if 'attention' in c]].describe().T)

,report_id,guidance_similarity_expanding_tfidf,guidance_novelty,guidance_novelty_expanding_tfidf,fulltext_novelty_expanding_tfidf
76,2025Q1,0.815826,0.184174,0.184174,0.093828
77,2025Q2,0.784184,0.215816,0.215816,0.099886
78,2025Q3,0.786610,0.213390,0.213390,0.109070
79,2025Q4,0.839593,0.160407,0.160407,0.103659
80,2026Q1,0.804280,0.195720,0.195720,0.115702


,count,mean,std,min,25%,50%,75%,max
guidance_attention_exchange_rate,80.0,8.912107,3.598407,0.000000,7.031278,8.160780,10.796033,18.224574
macro_attention_exchange_rate,80.0,0.937951,0.763863,0.000000,0.546382,0.810929,1.099600,5.100782
guidance_attention_financial_stability,80.0,3.116224,1.465458,0.000000,2.282285,3.066992,4.031557,6.329114
macro_attention_financial_stability,80.0,0.292937,0.783257,0.000000,0.000000,0.031372,0.242426,5.368861
guidance_attention_growth,80.0,13.031454,5.514848,6.170599,9.209940,12.272538,15.305887,38.048780
macro_attention_growth,80.0,20.451076,7.503202,2.866699,13.775603,19.732208,27.342991,33.444816
guidance_attention_inflation,80.0,1.007921,1.046043,0.000000,0.323185,0.834185,1.409074,5.947324
macro_attention_inflation,80.0,2.302350,2.249612,0.000000,1.098718,1.391398,2.578605,12.465022
guidance_attention_real_estate,80.0,1.712305,1.488234,0.000000,0.509938,1.397778,2.677308,5.576856
macro_attention_real_estate,80.0,2.276345,1.379755,0.000000,1.302339,2.103010,3.252819,6.269829


## 7. 股票事件级核心模型

In [8]:
stock_panel = pd.read_csv(ROOT / 'data/processed/refactor_stock_event_panel.csv')
stock_results = pd.read_csv(ROOT / 'output/results/stock_volatility_results.csv')
main_vol = json.loads((ROOT / 'output/results/stock_volatility_main.json').read_text(encoding='utf-8'))
display(stock_panel[['event_id','log_rv_0_5','guidance_novelty','pre_event_volatility_20d','action_nearby_core','post_2019']].tail())
display(stock_results)
print(main_vol['post_2019_total_effect'])

,event_id,log_rv_0_5,guidance_novelty,pre_event_volatility_20d,action_nearby_core,post_2019
75,2024Q4,-2.073027,0.218950,0.009014,0,1
76,2025Q1,-1.972477,0.184174,0.004322,1,1
77,2025Q2,-1.874082,0.215816,0.006295,0,1
78,2025Q3,-1.873230,0.213390,0.010488,0,1
79,2025Q4,-2.086470,0.160407,0.008039,0,1


,model,dependent,target,n,beta,se_hc3,p_value,r2,effect_percent_if_log_y
0,full,log_rv_0_5,guidance_novelty,79,0.746645,0.334847,0.025760,0.454702,110.990945
1,pre_2019,log_rv_0_5,guidance_novelty,50,0.049356,0.397946,0.901295,0.462765,5.059393
2,post_2019,log_rv_0_5,guidance_novelty,29,1.489204,1.435850,0.299662,0.242862,343.356437
3,covid,log_rv_0_5,guidance_novelty,12,-0.489641,6.494565,0.939902,0.160210,-38.715385
4,non_covid,log_rv_0_5,guidance_novelty,67,0.474270,0.364760,0.193524,0.493083,60.684087


{'estimate': 0.290985416460928, 'se_hc3': 0.6466179847624756, 'p_value': 0.6527022742493203, 'conf_int_95': [-0.976385833673524, 1.55835666659538]}


## 8. Student-t EGARCH-X 稳健性

In [9]:
egarch_x = json.loads((ROOT / 'output/results/daily_egarch_x_results.json').read_text(encoding='utf-8'))
main = egarch_x['main_model']
print({k: main.get(k) for k in ['method','sample_scope','date_start','date_end','n_daily_observations','n_report_events','n_novelty_events','n_policy_action_days','converged','runtime_seconds']})
print('parameters:', main.get('parameters'))
print('formal LR:', {k: main.get(k) for k in ['formal_lr_statistic','formal_lr_df','formal_lr_p_value','conditional_variance_change_pct_per_1sd_novelty','conditional_volatility_change_pct_per_1sd_novelty']})
display(pd.DataFrame(egarch_x['sensitivity']))
print('D0_D1 collinearity:', egarch_x['conditional_model'].get('D0_D1_collinearity_diagnostics'))
print('comparison:', egarch_x.get('comparison'))

{'method': 'full_joint_mle', 'sample_scope': 'full_continuous_daily_sequence', 'date_start': '2006-01-04', 'date_end': '2026-02-12', 'n_daily_observations': 4888, 'n_report_events': 80, 'n_novelty_events': 79, 'n_policy_action_days': 107, 'converged': True, 'runtime_seconds': 136.0214589999996}
parameters: {'mu_c': 0.046178612178507515, 'ar1': 0.004471174355038508, 'omega': 0.006072840658497743, 'alpha': 0.14372830509649764, 'gamma': -0.0056414608002087994, 'beta': 0.9912956084184092, 'report_day': -0.05789751253646343, 'novelty_z': 0.09444690550353113, 'policy_action_day': 0.048461820832572965, 'nu': 5.1842028408342165}
formal LR: {'formal_lr_statistic': 1.7271787231547933, 'formal_lr_df': 1, 'formal_lr_p_value': 0.18877160204997645, 'conditional_variance_change_pct_per_1sd_novelty': 9.905080803459022, 'conditional_volatility_change_pct_per_1sd_novelty': 4.835624099567903}


,date_window,method,nuisance_parameter_source,status,converged,nobs,restricted_loglik,unrestricted_loglik,conditional_loglik,conditional_loglik_gain,conditional_lr_statistic,conditional_lr_df,conditional_lr_p_value,optimizer_improvement_status,report_day_coef_fixed,policy_action_coef_fixed,exog_0_coef,exog_1_coef,exog_2_coef,exog_3_coef
0,D0,fixed_nuisance_conditional_likelihood,output/results/daily_egarch_x_locked.json,ok,True,4888,-8286.662222,-8285.690192,-8285.690192,0.972030,1.944061,1,0.163228,improved,-0.057898,0.048462,-0.057898,0.094330,0.048462,NaN
1,D1,fixed_nuisance_conditional_likelihood,output/results/daily_egarch_x_locked.json,ok,True,4888,-8286.662222,-8286.412287,-8286.412287,0.249935,0.499869,1,0.479558,improved,-0.057898,0.048462,-0.057898,0.048655,0.048462,NaN
2,D0_D1,fixed_nuisance_conditional_likelihood,output/results/daily_egarch_x_locked.json,ok,True,4888,-8286.662222,-8284.945242,-8284.945242,1.716980,3.433961,2,0.179608,improved,-0.057898,0.048462,-0.057898,0.264593,-0.192683,0.048462


D0_D1 collinearity: {'corr_novelty_d0_novelty_d1': 5.0568006744936646e-33, 'condition_number_of_event_design': 1.0, 'lambda_d0': 0.264593230905841, 'lambda_d1': -0.19268291846779137, 'lambda_d0_plus_lambda_d1': 0.07191031243804966, 'joint_lr_p_value': 0.17960765895061023, 'distributed_lag_collinearity_warning': False}
comparison: {'lambda_full_joint': 0.09444690550353113, 'lambda_conditional': 0.09433036951082434, 'sign_consistent': True, 'relative_difference': 0.0012338783582743823, 'loglik_full_joint': -8283.795644045806, 'conditional_loglik': -8285.690191703645, 'formal_lr_p_value': 0.18877160204997645, 'conditional_lr_p_value': 0.16322830761851442, 'beta_difference': 0.0, 'nu_difference': 0.0, 'significance_consistent': True, 'conditional_approximation_status': 'accepted'}


## 9. 市场功效分析

In [10]:
power_path = ROOT / 'output/diagnostics/market_power_analysis.csv'
if not power_path.exists():
    power_path = ROOT / 'output/tables/table8_market_power.csv'
power = pd.read_csv(power_path)
display(power)

,sample_size,power,observed_beta,min_detectable_effect,alpha,n_sim,method
0,79,0.6680,0.746645,0.938102,0.05,2000,wild_residual_bootstrap_hc3
1,100,0.7820,0.746645,0.833804,0.05,2000,wild_residual_bootstrap_hc3
2,120,0.8480,0.746645,0.761155,0.05,2000,wild_residual_bootstrap_hc3
3,160,0.9325,0.746645,0.659180,0.05,2000,wild_residual_bootstrap_hc3


## 10. 跨拟合政策语调与国债收益率曲线探索

In [11]:
bond = pd.read_csv(ROOT / 'output/results/cross_fitted_bond_exploration.csv')
yield_results = pd.read_csv(ROOT / 'output/results/yield_curve_results.csv')
display(yield_results)
display(bond)

,model,dependent,target,n,beta,se_hc3,p_value,r2,effect_percent_if_log_y,post_2019_interaction_beta,post_2019_total_effect,post_2019_total_p_value,bootstrap_ci_95_base,bootstrap_ci_95_total,permutation_p_base,permutation_p_interaction
0,main_yield_curve,delta_slope_bp_0_3,guidance_unexpected_tone,79,0.954200,1.185767,0.420987,0.058078,159.659209,-1.562208,-0.608008,0.723921,"[-0.9267454017622101, 3.654858620795652]","[-3.779766468098385, 1.2275147245224713]",0.383085,0.333333
1,secondary_yield_curve,delta_level_bp_0_3,guidance_unexpected_tone,79,0.541721,0.894063,0.544575,0.021241,71.896307,-0.970190,-0.428469,0.727257,"[-1.4856536651811287, 1.997540007238475]","[-3.223456656331668, 0.6077964970082458]",0.452736,0.407960
2,secondary_yield_curve,delta_curvature_bp_0_3,guidance_unexpected_tone,79,1.723496,1.493032,0.248353,0.085852,460.408352,-3.954519,-2.231024,0.018532,"[-1.3429637048100345, 3.964373069691203]","[-4.7313966657514115, 0.562955542773198]",0.074627,0.009950
3,auxiliary_1y_m1_p3,delta_yield_1y_bp_m1_p3,guidance_tone_change,80,-1.080867,1.912015,0.571868,0.031762,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,tone_aggregation,n,coef,p_value,interaction_coef,interaction_p_value,post_2019_total_effect,post_2019_total_p_value,r2
0,all_sentence_mean,76,3.067100,0.713106,1.297827,0.938676,4.364927,0.764339,0.052730
1,policy_relevant_mean,76,0.474421,0.907914,-0.586843,0.936024,-0.112422,0.985190,0.050897
2,directional_sentence_mean,76,1.966107,0.546610,4.415961,0.502897,6.382068,0.272453,0.064298


## 11. 图表、论文和提交包审计

In [12]:
summary = json.loads((ROOT / 'output/results/refactor_run_summary.json').read_text(encoding='utf-8')) if (ROOT / 'output/results/refactor_run_summary.json').exists() else {}
figures = sorted(p.name for p in (ROOT / 'output/figures').glob('figure*.png'))
manifest = pd.read_csv(ROOT / 'delivery/FINAL_SUBMISSION_MANIFEST.csv') if (ROOT / 'delivery/FINAL_SUBMISSION_MANIFEST.csv').exists() else pd.DataFrame()
print(summary)
print('figures:', figures)
print('final_submission_files:', len(manifest))

{'status': 'PASS', 'generated_at': '2026-06-21T14:36:12', 'main_volatility_beta': 0.7466450326474896, 'main_volatility_p': 0.025760080236270067, 'main_volatility_interaction_beta': -0.4556596161865616, 'main_volatility_post_2019_total_effect': {'estimate': 0.290985416460928, 'se_hc3': 0.6466179847624756, 'p_value': 0.6527022742493203, 'conf_int_95': [-0.976385833673524, 1.55835666659538]}, 'main_volatility_effect_percent': 110.9909451629524, 'manual_validation_rows': 240, 'text_validation_sentiment_accuracy': 0.29583333333333334, 'text_validation_topic_accuracy': 0.6541666666666667, 'sentiment_cv_accuracy': 0.8958333333333334, 'stance_cv_accuracy': 0.8583333333333333, 'egarch_x_status': 'full_joint_mle', 'egarch_x_novelty_coef': 0.0955076786677384, 'egarch_x_formal_lr_p': 0.18873709103066663, 'egarch_x_perm_p': 0.21, 'notebook_returncode': 0, 'notebook_executed_code_cells': 0, 'pdf_pages': 13, 'final_submission_files': 519, 'root_notebook_sha256': '28fa30e1fa2390db025a2fa508aa191a7bd36